In [3]:
from pathlib import Path
import qrunch as qc
import matplotlib as plt

# classical qchem modules
from pyscf import gto, scf, cc

import pyscf
from pyscf import mcscf, mp, fci
import numpy as np

from dmdm.interface import DMDM

# Add the license
qc.register_license_file("/home/flemming/Nextcloud/Cherimoya/training/master_cs/ms_project/qrunch/license_fm.txt")

## run vqe

In [ ]:
# """Run FAST-VQE on LiH as the first script."""
# Build molecular configuration for Lithium Hydride (LiH) molecule.

# example water:
    # 2.5369   -0.1550    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0
    # 3.0739    0.1550    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    # 2.0000    0.1550    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0


basis = "sto3g"

# m = [
#         ("H", 0.0, 0.0, 0.0),
#         ("Li", 1.5474, 0.0, 0.0),
#     ],

# # HO2
# m = [
#     ("O", 2.5369, -0.1550, 0.0000),
#     ("H", 3.0739, 0.1550, 0.0000),
#     ("H", 2.0000, 0.1550, 0.0000)

# ]

m = [
        ("O", 0.0, 0.0, 0.1035174918),
        ("H", 0.0, 0.7955612117, -0.4640237459),
        ("H", 0.0, -0.7955612117, -0.4640237459),
    ]

molecular_configuration = qc.build_molecular_configuration(
    molecule= m,
    basis_set=basis,
)

# Build the ground state problem.
problem_builder = qc.problem_builder_creator().ground_state().standard().create()
ground_state_problem = problem_builder.build_restricted(molecular_configuration)

fast_vqe_calculator = qc.calculator_creator().vqe().iterative().standard().create()

vqe_result = fast_vqe_calculator.calculate(ground_state_problem)

vqe_result.total_energy.value

-75.00164576895313

## prepare the integrals

In [7]:
mol = pyscf.M(atom = m, basis=basis)
myhf = mol.RHF().run()
mymp = mp.MP2(myhf).run()
noons, natorbs = mcscf.addons.make_natural_orbitals(mymp)
cisolver = fci.FCI(mol, natorbs)
e, fcivec = cisolver.kernel()
print(f"E(FCI) = {e}")

converged SCF energy = -74.9623593200333
E(RMP2) = -74.9988968573629  E_corr = -0.0365375373296453
E(SCS-RMP2) = -75.0044496453003  E_corr = -0.04209032526704
E(FCI) = -75.01375703590891


In [11]:
rdm1, rdm2, rdm3, rdm4 = fci.rdm.make_dm1234('FCI4pdm_kern_sf', fcivec, fcivec, 4, 4)
rdm1, rdm2, rdm3, rdm4 = fci.rdm.reorder_dm1234(rdm1, rdm2, rdm3, rdm4)

AssertionError: 

In [12]:
import numpy as np
from pyscf import gto, scf, mp, mcscf, fci, ao2mo

# ==========================================
# 1. Define Molecule
# ==========================================
# Water geometry: O at origin, H atoms at ~0.96 Angstrom, angle 104.5 deg
# Using standard bond length and angle
mol = gto.M(
    atom='''
    O  0.000000  0.000000  0.000000
    H  0.000000  0.756950  0.586100
    H  0.000000 -0.756950  0.586100
    ''',
    basis='cc-pvdz',  # Double-zeta basis set
    unit='Angstrom',
    symmetry=False,   # Disable symmetry for easier debugging (optional)
    verbose=4         # Print detailed output
)

print(f"Molecule: {mol.natm} atoms, {mol.nelectron} electrons, {mol.nao} basis functions")

# ==========================================
# 2. Hartree-Fock Reference
# ==========================================
mf = scf.RHF(mol).run()
print(f"\n--- RHF Energy: {mf.e_tot:.8f} Ha ---")

# ==========================================
# 3. MP2 for Natural Orbitals
# ==========================================
# MP2 provides better correlation info for generating Natural Orbitals
mp2 = mp.MP2(mf).run()
print(f"MP2 Correlation Energy: {mp2.e_corr:.8f} Ha")

# Generate Natural Orbitals (NOs)
# noons: Natural Orbital Occupation Numbers
# natorbs: Coefficients of Natural Orbitals
noons, natorbs = mcscf.addons.make_natural_orbitals(mp2)

# Sort NOs by occupation number (descending) just to be safe
idx = np.argsort(noons)[::-1]
noons = noons[idx]
natorbs = natorbs[:, idx]

print(f"\n--- Natural Orbital Occupations (Top 10) ---")
for i, occ in enumerate(noons[:10]):
    print(f"Orbital {i}: {occ:.4f}")

# ==========================================
# 4. Define Active Space (CAS)
# ==========================================
# For H2O with cc-pVDZ:
# Total Electrons: 10 (8 valence + 2 core)
# Core: 1s of Oxygen (2 electrons) -> Frozen
# Valence: 8 electrons.
# Active Space: Usually the 4 valence orbitals (2 bonding, 2 lone pairs) containing 8 electrons?
# Or a smaller active space like CAS(6,4) if we freeze the O 1s and maybe one bonding pair?
# Let's do a standard CAS(6,4): 6 active electrons in 4 active orbitals.
# This typically corresponds to the 2 lone pairs and the 2 O-H bonding orbitals, 
# but we might freeze the deepest bonding orbital or treat it differently.
# 
# Strategy: Take the 4 orbitals closest to the Fermi level (occupation ~2 or ~0) 
# that are chemically relevant. 
# For H2O, the valence shell is 4 orbitals (1a1, 1b2, 3a1, 1b1 in C2v).
# Let's select the 4 orbitals with occupations closest to 2.0 (but not the core 1s).
# Since we sorted by occupation, the core 1s will be at the top (~2.0).
# We skip the core and take the next 4.

n_core = 1  # Freeze O 1s (2 electrons)
n_active = 4 # 4 active orbitals
n_active_elec = 6 # 6 active electrons (Total 8 valence - 2 frozen = 6? Or just 6 in the active space)
# Let's assume we want to correlate 6 electrons in 4 orbitals.
# Total electrons = 10. 
# If we freeze 1s (2e), we have 8 valence. 
# If we pick 4 orbitals, we can put 6 or 8 electrons. 
# Let's do CAS(6,4) which is a common test case.

# Select indices for active orbitals
# Skip the first 'n_core' (which are the 1s core)
# Then take the next 'n_active'
start_idx = n_core
end_idx = n_core + n_active
active_indices = list(range(start_idx, end_idx))

print(f"\n--- Active Space Selection ---")
print(f"Core orbitals frozen: {n_core}")
print(f"Active orbitals selected: {active_indices}")
print(f"Active electrons: {n_active_elec}")

# Slice the NO coefficients
natorbs_active = natorbs[:, active_indices]
print(f"Active NO shape: {natorbs_active.shape}")

# ==========================================
# 5. Transform Integrals to Active Space
# ==========================================
# Get 1-electron integrals (Kinetic + Nuclear)
h1 = mol.intor('int1e_kin') + mol.intor('int1e_nuc')
# Transform to active MO basis
h_mo = ao2mo.incore(h1, natorbs_active) # Shape: (n_active, n_active)

# Get 2-electron integrals
# Note: mol.intor('int2e') returns the full AO 4-index tensor
# We transform only the active subset
g_mo = ao2mo.incore(mol.intor('int2e'), natorbs_active) # Shape: (n_active, n_active, n_active, n_active)

print(f"\n--- Transformed Integrals ---")
print(f"h_mo shape: {h_mo.shape}")
print(f"g_mo shape: {g_mo.shape}")

# ==========================================
# 6. Run FCI in Active Space
# ==========================================
# Create FCI solver
# We pass the molecule (for metadata) and the number of active electrons
cisolver = fci.FCI(mol, n_active_elec)
cisolver.norb = n_active # Explicitly set the number of orbitals in the active space

# Run the solver
fci_energy, fcivec = cisolver.kernel()

print(f"\n--- FCI Results ---")
print(f"FCI Energy: {fci_energy:.8f} Ha")
print(f"FCI Vector Norm: {np.linalg.norm(fcivec):.6f}")

# ==========================================
# 7. Compute Reduced Density Matrices (RDMs)
# ==========================================
# Generate 1-RDM and 2-RDM
# make_dm1234 returns (dm1, dm2, dm3, dm4)
# Arguments: method, vec, vec, norb, nelec
dm1, dm2, dm3, dm4 = fci.rdm.make_dm1234(
    'FCI4pdm_kern_sf', 
    fcivec, 
    fcivec, 
    n_active, 
    n_active_elec
)

# Reorder if necessary (PySCF usually returns in standard order, but good to be safe)
# reorder_dm1234 swaps indices to match standard convention (i,j,k,l) -> (i,j,k,l)
dm1, dm2, dm3, dm4 = fci.rdm.reorder_dm1234(dm1, dm2, dm3, dm4)

print(f"\n--- RDM Shapes ---")
print(f"1-RDM: {dm1.shape}")
print(f"2-RDM: {dm2.shape}")

# ==========================================
# 8. Compute Properties (e.g., Dipole Moment)
# ==========================================
# Dipole moment = <Psi | mu | Psi> = Tr(DM1 * mu_integrals)
# Get dipole integrals in AO basis
dip_ao = mol.intor('int1e_r') # Shape: (3, nao, nao) -> x, y, z components

# Transform dipole integrals to active MO basis
# We need to transform the 3 components
dip_mo = np.zeros((3, n_active, n_active))
for i in range(3):
    dip_mo[i] = ao2mo.incore(dip_ao[i], natorbs_active)

# Calculate dipole moment components
dip_moment = np.zeros(3)
for i in range(3):
    # Trace: sum_{pq} DM1[p,q] * mu[p,q]
    dip_moment[i] = np.sum(dm1 * dip_mo[i])

# Add nuclear contribution
# Nuclear charge * position
nuc_dip = np.zeros(3)
for i in range(mol.natm):
    Z = mol.atom_charge(i)
    pos = mol.atom_coord(i)
    nuc_dip += Z * pos

total_dip = dip_moment + nuc_dip
dip_magnitude = np.linalg.norm(total_dip)

print(f"\n--- Dipole Moment (Debye) ---")
print(f"Electronic contribution (a.u.): {dip_moment}")
print(f"Nuclear contribution (a.u.): {nuc_dip}")
print(f"Total Dipole (a.u.): {total_dip}")
print(f"Total Dipole (Debye): {dip_magnitude * 2.54175:.4f} D")

# ==========================================
# 9. (Optional) Custom DMDM Class Integration
# ==========================================
# If you have your custom DMDM class, here is how you would call it:
# Assuming DMDM is imported from your local file
# from my_module import DMDM 

# dmdm_instance = DMDM(
#     h_mo,
#     g_mo,
#     0, # spin
#     n_active,
#     0, # ?
#     n_active_elec // 2, # alpha electrons (assuming closed shell)
#     dm1,
#     rdm2=dm2,
#     rdm3=dm3,
#     rdm4=dm4
# )
# result = dmdm_instance.run()

print("\nWorkflow completed successfully.")

System: uname_result(system='Linux', node='cachyos', release='6.19.6-2-cachyos', version='#1 SMP PREEMPT_DYNAMIC Fri, 06 Mar 2026 11:29:20 +0000', machine='x86_64')  Threads 6
Python 3.11.14 | packaged by conda-forge | (main, Jan 26 2026, 23:48:32) [GCC 14.3.0]
numpy 2.4.3  scipy 1.15.3  h5py 3.15.1
Date: Mon Mar 30 19:20:22 2026
PySCF version 2.12.1
PySCF path  /home/flemming/micromamba/envs/qrunch/lib/python3.11/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 3
[INPUT] num. electrons = 10
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = Angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000000000   0.000000000000   0.000000000000 AA    0.000000000000   0.000000000000   0.000000000000 Bohr   0.0
[INPUT]  2 H      0.000000000000   0.756950000000   0.586100000000 AA 

TypeError: 'module' object is not callable